<a href="https://colab.research.google.com/github/SNK0-0/Bit-Net-LLM_Proj/blob/main/BitNet_LLM_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ------------------------------------------------------------------------------
# STARTUP CHECKLIST FOR EVERY NEW SESSION (Your 2-minute setup routine)
# ------------------------------------------------------------------------------
# [ ] SET RUNTIME: Go to Runtime -> Change runtime type.
#     -> Choose CPU for writing code and debugging layer architectures.
#     -> Choose T4 GPU only when ready to initiate an active training execution cell.
# [ ] LAUNCH ENVIRONMENT: Run Cell 1 (Mounts Drive, clones fresh repo, installs pip, builds tokens).
# [ ] REGENERATE CUSTOM CODE: Run all your `%%writefile` cells to inject modules into the VM.
# [ ] ROUTE & RUN: Fire off your training loop with output paths pointing directly to your Drive.
# ==============================================================================

In [5]:
# 1. Mount your permanent storage
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the repository into the volatile VM storage
!git clone https://github.com/karpathy/nanoGPT.git
%cd nanoGPT

# 3. Install required libraries
!pip install torch numpy transformers datasets tiktoken wandb tqdm

# 4. Prepare the dataset tokens
!python data/shakespeare_char/prepare.py

Mounted at /content/drive
Cloning into 'nanoGPT'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.25 KiB | 20.88 MiB/s, done.
Resolving deltas: 100% (380/380), done.
/content/nanoGPT
length of dataset in characters: 1,115,394
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


In [ ]:
!python train.py config/train_shakespeare_char.py --out_dir='/content/drive/MyDrive/BitNet_Project/baseline' --max_iters=1500

Overriding config with config/train_shakespeare_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of 

In [ ]:
# Final train loss : 1.1510
# Final validation loss : 1.4739
# avg. iteration time : 570-580 ms
# Model flops utilization : 0.63%-0.64%

In [6]:
%%writefile bitlinear.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class BitLinear(nn.Linear):
    """
    Custom Linear layer for BitNet b1.58.
    Quantizes weights to ternary (-1, 0, 1) and activations to 8-bit (int8).
    """
    def __init__(self, in_features, out_features, bias=False, eps=1e-5):
        # We inherit from nn.Linear so self.weight and self.bias are handled automatically
        super().__init__(in_features, out_features, bias)
        self.eps = eps

    def forward(self, x):
        # -------------------------------------------------------------
        # 1. Weight Quantization (Ternary)
        # -------------------------------------------------------------
        w = self.weight

        # Calculate gamma: the mean absolute value of the weight matrix
        gamma = w.abs().mean()

        # Scale, round, and clamp to ternary range [-1, 0, 1]
        w_scaled = w / (gamma + self.eps)
        w_quant = torch.clamp(torch.round(w_scaled), -1, 1)

        # Straight-Through Estimator (STE) trick for weights
        # Forward pass uses w_quant, backward pass updates original w
        w_quant = w + (w_quant - w).detach()

        # -------------------------------------------------------------
        # 2. Activation Quantization (8-bit)
        # -------------------------------------------------------------
        # Calculate beta: the max absolute value per token/feature
        beta = x.abs().max(dim=-1, keepdim=True).values

        # Scale to 8-bit integer range [-128, 127], round, and clamp
        x_scaled = x * (127.0 / (beta + self.eps))
        x_quant = torch.clamp(torch.round(x_scaled), -128, 127)

        # Straight-Through Estimator (STE) trick for activations
        x_quant = x + (x_quant - x).detach()

        # -------------------------------------------------------------
        # 3. The Core Matrix Multiplication
        # -------------------------------------------------------------
        # F.linear performs: x_quant @ w_quant^T + bias
        output = F.linear(x_quant, w_quant, self.bias)

        # -------------------------------------------------------------
        # 4. Dequantization / Rescaling
        # -------------------------------------------------------------
        # We divided w by gamma and x by (beta/127), so we must multiply them back
        scale_factor = (gamma * beta) / 127.0
        output = output * scale_factor

        return output

Writing bitlinear.py


In [7]:
# Create a new cell and run this to generate model_bitnet.py
import os

# Read the original nanoGPT model architecture
with open('model.py', 'r') as f:
    model_code = f.read()

# Inject our custom module import
bitnet_imports = """
import math
import inspect
from bitlinear import BitLinear
"""

# Define the targeted Attention layer replacements
old_attention = """        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)"""

new_attention = """        # key, query, value projections for all heads using BitLinear
        self.c_attn = BitLinear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection using BitLinear
        self.c_proj = BitLinear(config.n_embd, config.n_embd, bias=config.bias)"""

# Define the targeted MLP layer replacements
old_mlp = """        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)"""

new_mlp = """        self.c_fc    = BitLinear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = BitLinear(4 * config.n_embd, config.n_embd, bias=config.bias)"""

# Apply transformations and write to disk
modified_code = model_code.replace("import math\nimport inspect", bitnet_imports)
modified_code = modified_code.replace(old_attention, new_attention)
modified_code = modified_code.replace(old_mlp, new_mlp)

with open('model_bitnet.py', 'w') as f:
    f.write(modified_code)

print("model_bitnet.py successfully generated with BitLinear modules injected.")

model_bitnet.py successfully generated with BitLinear modules injected.


In [8]:
# Create a new cell and run this to generate train_bitnet.py
import os

# Read the original nanoGPT training script
with open('train.py', 'r') as f:
    train_script = f.read()

# Reroute the import architecture
modified_train = train_script.replace(
    "from model import GPTConfig, GPT",
    "from model_bitnet import GPTConfig, GPT"
)

# Write out the new executable engine
with open('train_bitnet.py', 'w') as f:
    f.write(modified_train)

print("train_bitnet.py engine complete and ready to deploy.")

train_bitnet.py engine complete and ready to deploy.


In [ ]:
!python train_bitnet.py config/train_shakespeare_char.py --out_dir='/content/drive/MyDrive/BitNet_project/quantized_run' --max_iters=1500

Overriding config with config/train_shakespeare_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of 

In [ ]:
# ==============================================================================
# PHASE 5 RESULTS: 1.58-BIT QUANTIZED RUN (DEV B TRACK)
# ==============================================================================
# Final Train Loss (Step 1500): 2.4712
# Final Val Loss   (Step 1500): 2.4870
# Model MFU                   : 0.47%
#
# NOTES:
# - Checkpoint successfully routed to: /content/drive/MyDrive/bitnet_project/quantized_run
# - Val loss is higher than FP16 baseline (expected without hyperparameter tuning).
# - MFU is lower because the custom BitLinear layer relies on unoptimized Python ops
#   rather than custom CUDA kernels. Mathematical logic is verified and functioning.
# ==============================================================================

In [9]:
%%writefile generate.py
import torch
import pickle
import os
from model_bitnet import GPT, GPTConfig

# 1. Hardware and Path Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# LOCKED TO YOUR PREFERRED DIRECTORY NAME
out_dir = '/content/drive/MyDrive/BitNet_Project/quantized_run_5k'
ckpt_path = os.path.join(out_dir, 'ckpt.pt')

if not os.path.exists(ckpt_path):
    print(f"❌ Error: Checkpoint not found at {ckpt_path}.")
else:
    print(f"Loading BitNet b1.58 model from {ckpt_path}...")
    checkpoint = torch.load(ckpt_path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)

    # 2. THE FIX: Strip the '_orig_mod.' prefix from compiled weights
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'

    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    # Load the cleaned dictionary into the model
    model.load_state_dict(state_dict)

    model.eval()
    model.to(device)

    # 3. Load the Shakespeare Character Vocabulary
    meta_path = os.path.join('data', 'shakespeare_char', 'meta.pkl')
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    stoi, itos = meta['stoi'], meta['itos']

    # 4. Set the Prompt
    prompt = "O Romeo"
    x = torch.tensor([stoi[c] for c in prompt], dtype=torch.long, device=device).unsqueeze(0)

    # 5. Generate Text!
    print("\n--- Generating Text from 1.58-Bit Model (Strict Constants) ---\n")
    with torch.no_grad():
        # Lowered temperature to 0.2 and added top_k=10 to eliminate noise
        y = model.generate(x, max_new_tokens=500, temperature=0.65, top_k=8)
        generated_text = ''.join([itos[int(i)] for i in y[0]])
        print(generated_text)

Writing generate.py


In [11]:
!python generate.py

Loading BitNet b1.58 model from /content/drive/MyDrive/BitNet_Project/quantized_run_5k/ckpt.pt...
number of parameters: 10.65M

--- Generating Text from 1.58-Bit Model (Strict Constants) ---

O Romeon ce me ban stiee and the mes
For thallll sor t blor ot,
He thor my athe he thalour t s he myore than hand the ourdinerice brs ant the
I he the tond satile ay se my thaneeare thinco therenghant thery there an ald
Whand the sould basen thuco t the s bo wowiee thare thes,
Herd at he he her thenomy ar ou t wes he wen then heen t mus the thim sor t theane the t seran the and thar h we thoor hand s me sthoulen this buthe br my ble
Thise winthather t o hano sthean ay the a anou ar my shertowithe hasou


In [ ]:
!python train_bitnet.py config/train_shakespeare_char.py \
    --out_dir='/content/drive/MyDrive/BitNet_Project/quantized_run_5k' \
    --max_iters=5000 \
    --learning_rate=3e-3 \
    --min_lr=3e-4

Overriding config with config/train_shakespeare_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of 

In [ ]:
import os
os.sync()

In [10]:
%%writefile configuration_bitnet.py
from transformers import PretrainedConfig

class BitNetGPTConfig(PretrainedConfig):
    """
    Configuration class for Bit-Net-LLM_Project.
    Inherits from PretrainedConfig for seamless Hugging Face serialization.
    """
    model_type = "bitnet_gpt"

    def __init__(
        self,
        vocab_size=65,
        block_size=256,
        n_layer=6,
        n_head=6,
        n_embd=384,
        dropout=0.2,
        bias=False,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        self.bias = bias

Writing configuration_bitnet.py


In [11]:
%%writefile modeling_bitnet.py
import torch
import torch.nn as nn
from transformers import PreTrainedModel
from configuration_bitnet import BitNetGPTConfig
from bitlinear import BitLinear
from model_bitnet import Block

class BitNetGPTPreTrainedModel(PreTrainedModel):
    config_class = BitNetGPTConfig
    base_model_prefix = "transformer"

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, BitLinear)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.04)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

class BitNetGPTForCausalLM(BitNetGPTPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            # THE FIX: Added bias=config.bias to dynamically match your checkpoint
            ln_f = nn.LayerNorm(config.n_embd, bias=config.bias)
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.post_init()

    def forward(self, input_ids, labels=None):
        pass

Writing modeling_bitnet.py


In [12]:
%%writefile export_to_hf.py
import torch
import os
from configuration_bitnet import BitNetGPTConfig
from modeling_bitnet import BitNetGPTForCausalLM

# 1. Setup Paths
device = 'cpu'
out_dir = '/content/drive/MyDrive/BitNet_Project/quantized_run_5k'
ckpt_path = os.path.join(out_dir, 'ckpt.pt')
hf_export_dir = '/content/drive/MyDrive/BitNet_Project/hf_bitnet_model'

# 2. Load the Raw PyTorch Checkpoint
print(f"Loading checkpoint from {ckpt_path}...")
checkpoint = torch.load(ckpt_path, map_location=device)

# Clean the PyTorch 2.0 compilation prefix
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

# 3. Create Hugging Face Config and Model instances
model_args = checkpoint['model_args']
hf_config = BitNetGPTConfig(**model_args)
hf_model = BitNetGPTForCausalLM(hf_config)

# 4. Inject Weights into the HF Wrapper
hf_model.load_state_dict(state_dict)

# 5. Export using Hugging Face format (.save_pretrained)
print(f"Saving Hugging Face model to {hf_export_dir}...")
hf_model.save_pretrained(hf_export_dir, safe_serialization=True)
print("✅ Success! Your 1.58-bit model is now wrapped in the Hugging Face ecosystem.")

Writing export_to_hf.py


In [17]:
!python export_to_hf.py

Loading checkpoint from /content/drive/MyDrive/BitNet_Project/quantized_run_5k/ckpt.pt...
Saving Hugging Face model to /content/drive/MyDrive/BitNet_Project/hf_bitnet_model...
Writing model shards: 100% 1/1 [00:00<00:00, 12.83it/s]
✅ Success! Your 1.58-bit model is now wrapped in the Hugging Face ecosystem.


In [14]:
from huggingface_hub import login
from modeling_bitnet import BitNetGPTForCausalLM

# 1. Authenticate Securely
print("Requesting Hugging Face Authentication...")
login()

# 2. Load the Local Weights
hf_export_dir = '/content/drive/MyDrive/BitNet_Project/hf_bitnet_model'
print("Loading local 1.58-bit Hugging Face model...")
hf_model = BitNetGPTForCausalLM.from_pretrained(hf_export_dir)

# 3. Push to the Hub
print("Pushing model to the Hub...")
hf_model.push_to_hub("Mudunk/BitNet_LLM_Project")

print("✅ Success! The model is now live on the open-source Hub.")

Requesting Hugging Face Authentication...
Loading local 1.58-bit Hugging Face model...


Loading weights:   0%|          | 0/40 [00:00<?, ?it/s]

Pushing model to the Hub...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...16n1t_u/model.safetensors:   1%|1         |  565kB / 43.1MB            

✅ Success! The model is now live on the open-source Hub.
